# 1. TF-IDF

## Ручное построение

In [ ]:
import math
import numpy as np

In [ ]:
docs = [
    "Cats chase mice",
    "Dogs chase cats",
    "Mice love cheese"
]

Шаг 1: Токенизация предложений

In [ ]:
tokenized_docs = [[word.lower() for word in sentence.split()] for sentence in docs]
tokenized_docs

[['cats', 'chase', 'mice'],
 ['dogs', 'chase', 'cats'],
 ['mice', 'love', 'cheese']]

Шаг 2: Построение словаря

In [ ]:
vocab = sorted(set(word for doc in tokenized_docs for word in doc))
print("Vocabulary:", vocab)

Vocabulary: ['cats', 'chase', 'cheese', 'dogs', 'love', 'mice']


Шаг 3: Частота слова (term frequency, TF)

In [ ]:
tf = []
for doc in tokenized_docs:
    tf_vec = [doc.count(word) / len(doc) for word in vocab]
    tf.append(tf_vec)
print("TF:\n", np.array(tf))

TF:
 [[0.33333333 0.33333333 0.         0.         0.         0.33333333]
 [0.33333333 0.33333333 0.         0.33333333 0.         0.        ]
 [0.         0.         0.33333333 0.         0.33333333 0.33333333]]


Шаг 4: Частота документа (document frequency, DF) и обратная частота (inverse document frequency, IDF)

In [ ]:
N = len(tokenized_docs)
df = [sum(1 for doc in tokenized_docs if word in doc) for word in vocab]
#idf = [math.log(N/df_i) for df_i in df]
idf = [math.log((1+N)/(1+df_i))+1 for df_i in df]
print("IDF:", np.round(idf, 2))

IDF: [1.29 1.29 1.69 1.69 1.69 1.29]


Шаг 5: TF x IDF

In [ ]:
tfidf = []
for tf_vec in tf:
    tfidf_vec = [tf_vec[i] * idf[i] for i in range(len(vocab))]
    tfidf.append(tfidf_vec)

tfidf = np.array(tfidf)
print("TF-IDF matrix:\n", np.round(tfidf, 2))

TF-IDF matrix:
 [[0.43 0.43 0.   0.   0.   0.43]
 [0.43 0.43 0.   0.56 0.   0.  ]
 [0.   0.   0.56 0.   0.56 0.43]]


Шаг 6: Косинусная близость

$ cos\_sim = \frac{x \cdot y}{||x|| \cdot ||y||} $

In [ ]:
cos_sim = np.dot(tfidf[0], tfidf[1]) / (np.linalg.norm(tfidf[0]) * np.linalg.norm(tfidf[1]))
print("Cosine similarity (Doc1 vs Doc2):", round(cos_sim, 2))

Cosine similarity (Doc1 vs Doc2): 0.6


## С помощью sklearn

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
docs = [
    "Cats chase mice",
    "Dogs chase cats",
    "Mice love cheese"
]

Строим матрицу TF-IDF

In [ ]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(docs)

In [ ]:
type(X)

scipy.sparse._csr.csr_matrix

In [ ]:
print("Features:", vectorizer.get_feature_names_out())
print("TF-IDF Matrix:\n", X.toarray())

Features: ['cats' 'chase' 'cheese' 'dogs' 'love' 'mice']
TF-IDF Matrix:
 [[0.57735027 0.57735027 0.         0.         0.         0.57735027]
 [0.51785612 0.51785612 0.         0.68091856 0.         0.        ]
 [0.         0.         0.62276601 0.         0.62276601 0.4736296 ]]


Косинусная близость

In [ ]:
cos_sim = cosine_similarity(X, X)
print("Cosine Similarity:\n", np.round(cos_sim, 2))

Cosine Similarity:
 [[1.   0.6  0.27]
 [0.6  1.   0.  ]
 [0.27 0.   1.  ]]


# 2. Word2Vec

## Ручное построение

Шаг 1: Создаем список слов

In [ ]:
sentence = ["cats", "chase", "mice", "and", "dogs", "chase", "cats"]
window = 2 # Гиперпараметр

Шаг 2: Строим словарь

In [ ]:
vocab = list(set(sentence))
word2idx = {w: i for i, w in enumerate(vocab)}
print("Vocab index:", word2idx)

Vocab index: {'chase': 0, 'mice': 1, 'and': 2, 'dogs': 3, 'cats': 4}


Шаг 3: Создаем матрицу совместного появления слов

In [ ]:
cooc = np.zeros((len(vocab), len(vocab)))
for i, word in enumerate(sentence):
    idx = word2idx[word]
    for j in range(max(0, i-window), min(len(sentence), i+window+1)):
        if i != j:
            context_word = sentence[j]
            cooc[idx, word2idx[context_word]] += 1

print("Co-occurrence matrix:\n", cooc)

Co-occurrence matrix:
 [[0. 1. 2. 1. 2.]
 [1. 0. 1. 1. 1.]
 [2. 1. 0. 1. 0.]
 [1. 1. 1. 0. 1.]
 [2. 1. 0. 1. 0.]]


Шаг 4: Нормализуем строки (создание базовых эмбеддингов)

In [ ]:
embeddings = cooc / (cooc.sum(axis=1, keepdims=True) + 1e-9)
print("Embeddings:\n", np.round(embeddings, 2))

Embeddings:
 [[0.   0.17 0.33 0.17 0.33]
 [0.25 0.   0.25 0.25 0.25]
 [0.5  0.25 0.   0.25 0.  ]
 [0.25 0.25 0.25 0.   0.25]
 [0.5  0.25 0.   0.25 0.  ]]


Шаг 5: Похожесть между "cats" и "mice"

In [ ]:
vec_cats = embeddings[word2idx["cats"]]
vec_mice = embeddings[word2idx["mice"]]
sim = np.dot(vec_cats, vec_mice) / (np.linalg.norm(vec_cats) * np.linalg.norm(vec_mice))
print("Cosine similarity cats vs mice:", round(sim, 2))

Cosine similarity cats vs mice: 0.61


## С помощью Gensim

In [ ]:
! pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 44.6 MB/s eta 0:00:00


In [ ]:
from gensim.models import Word2Vec

Данные для обучения

In [ ]:
sentences = [
    ["cats", "chase", "mice"],
    ["dogs", "chase", "cats"],
    ["mice", "love", "cheese"],
    ["rats", "love", "cheese"],
    ["dogs", "bark", "loudly"]
]

Обучение простой Word2Vec модели

In [ ]:
model = Word2Vec(sentences, vector_size = 10, window = 2, min_count = 1, sg = 1)

Проверяем схожесть между словами

In [ ]:
print("Similarity cats-mice:", model.wv.similarity("cats", "mice"))
print("Similarity mice-cheese:", model.wv.similarity("mice", "cheese"))
print("Similarity cats-cheese:", model.wv.similarity("cats", "cheese"))

Similarity cats-mice: 0.09267305
Similarity mice-cheese: 0.10494352
Similarity cats-cheese: -0.27260208


Вектор схожести для слова cats

In [ ]:
print("Vector for 'cats':\n", model.wv["cats"])

Vector for 'cats':
 [-0.08157917  0.04495798 -0.04137076  0.00824536  0.08498619 -0.04462177
  0.045175   -0.0678696  -0.03548489  0.09398508]


In [ ]:
vec_cats = model.wv["cats"]
vec_mice = model.wv["mice"]
sim = np.dot(vec_cats, vec_mice) / (np.linalg.norm(vec_cats) * np.linalg.norm(vec_mice))
print("Cosine similarity cats vs mice:", round(sim, 3))

Cosine similarity cats vs mice: 0.093


## Кластеризация документов

In [ ]:
!pip install pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 87.4 MB/s eta 0:00:00


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pymorphy3
import nltk
from nltk.corpus import stopwords

In [ ]:
docs = [
    "Кошки ловят мышей во дворе",
    "Собаки гоняются за кошками",
    "Мыши любят сыр",
    "Политическая ситуация в России стабильная",
    "В экономике России наблюдается рост",
]

In [ ]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
russian_stopwords = stopwords.words("russian")

In [ ]:
morph = pymorphy3.MorphAnalyzer()

def tokenize(text):
    tokens = [morph.parse(word)[0].normal_form for word in text.lower().split() if word not in russian_stopwords]
    return tokens

tokenized_docs = [tokenize(doc) for doc in docs]
print("Tokenized:", tokenized_docs)

Tokenized: [['кошка', 'ловить', 'мышь', 'двор'], ['собака', 'гоняться', 'кошка'], ['мышь', 'любить', 'сыр'], ['политический', 'ситуация', 'россия', 'стабильный'], ['экономика', 'россия', 'наблюдаться', 'рост']]


С помощью TF-IDF

In [ ]:
vectorizer = TfidfVectorizer(tokenizer=lambda x: x, lowercase=False)
X = vectorizer.fit_transform(tokenized_docs)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [ ]:
import pandas as pd
pd.DataFrame(np.round(X.toarray(), 2), columns = vectorizer.get_feature_names_out())

,гоняться,двор,кошка,ловить,любить,мышь,наблюдаться,политический,россия,рост,ситуация,собака,стабильный,сыр,экономика
0,0.00,0.55,0.44,0.55,0.00,0.44,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,0.61,0.00,0.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.61,0.00,0.00,0.00
2,0.00,0.00,0.00,0.00,0.61,0.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.61,0.00
3,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.52,0.42,0.00,0.52,0.00,0.52,0.00,0.00
4,0.00,0.00,0.00,0.00,0.00,0.00,0.52,0.00,0.42,0.52,0.00,0.00,0.00,0.00,0.52


In [ ]:
cos_sim = cosine_similarity(X, X)
print("\nTF-IDF Cosine Similarities:\n", np.round(cos_sim, 2))


TF-IDF Cosine Similarities:
 [[1.   0.22 0.22 0.   0.  ]
 [0.22 1.   0.   0.   0.  ]
 [0.22 0.   1.   0.   0.  ]
 [0.   0.   0.   1.   0.18]
 [0.   0.   0.   0.18 1.  ]]


С помощью Word2Vec

In [ ]:
model = Word2Vec(sentences = tokenized_docs, vector_size=20, window=2, min_count=1, sg=1)

In [ ]:
def doc_vector(doc):
    vecs = [model.wv[w] for w in doc if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size)

doc_vecs = np.array([doc_vector(doc) for doc in tokenized_docs])

In [ ]:
cos_sim_w2v = cosine_similarity(doc_vecs, doc_vecs)
print("\nWord2Vec Document Similarities:\n", np.round(cos_sim_w2v, 2))


Word2Vec Document Similarities:
 [[ 1.   -0.07  0.47 -0.19  0.02]
 [-0.07  1.    0.11  0.05  0.06]
 [ 0.47  0.11  1.   -0.17  0.03]
 [-0.19  0.05 -0.17  1.    0.18]
 [ 0.02  0.06  0.03  0.18  1.  ]]


## Поиск схожих по смыслу предложений

TF-IDF

In [ ]:
def tfidf_search(query, topn = 2):
    q_tokens = tokenize(query)
    q_vec = vectorizer.transform([q_tokens])
    sims = cosine_similarity(q_vec, X).flatten()
    top_idx = sims.argsort()[::-1][:topn]
    return [(docs[i], round(float(sims[i]), 3)) for i in top_idx]

In [ ]:
tfidf_search("Собаки любят гоняться за кошками")

[('Собаки гоняются за кошками', 0.852), ('Мыши любят сыр', 0.321)]

In [ ]:
tfidf_search("Давайте поговорим об экономике")

[('В экономике России наблюдается рост', 0.523),
 ('Политическая ситуация в России стабильная', 0.0)]

Word2Vec

In [ ]:
model = Word2Vec(sentences=tokenized_docs, vector_size = 20, window = 2, min_count = 1, sg=1)

In [ ]:
def doc_vector(doc):
    vecs = [model.wv[w] for w in doc if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size)

doc_vecs = np.array([doc_vector(doc) for doc in tokenized_docs])

In [ ]:
def w2v_search(query, topn = 2):
    q_tokens = tokenize(query)
    q_vec = doc_vector(q_tokens).reshape(1, -1)
    sims = cosine_similarity(q_vec, doc_vecs).flatten()
    top_idx = sims.argsort()[::-1][:topn]
    return [(docs[i], round(float(sims[i]), 3)) for i in top_idx]

In [ ]:
w2v_search("Собаки любят гоняться за кошками")

[('Собаки гоняются за кошками', 0.877), ('Мыши любят сыр', 0.412)]

In [ ]:
w2v_search("Давайте поговорим об экономике")

[('В экономике России наблюдается рост', 0.602),
 ('Политическая ситуация в России стабильная', 0.212)]